# Order Estimation with simulated data

Simulate data, fit different HMMs with a varying number of hidden states. Compute BIC and AIC, explore dependence on sample size and difference between AR(1) model parameters.

## Imports

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))
from methods.hmm_ar_1_k_states import (
    simulate_rs_ar1,
    transform_params,
    obs_density,
    forward_algorithm,
    neg_loglik,
    fit_model,
    filtered_probs
)
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
def fit_model_robust(y, K, true_beta=None, true_sigma=None, true_P=None, n_starts=3):
    best_loglik = -np.inf
    best_result = None
    best_params = None
    
    for start in range(n_starts):
        if K == 2 and true_beta is not None:
            beta0 = np.log((1 + true_beta) / (1 - true_beta)) + np.random.normal(scale=0.1, size=2)
            sigma0 = np.log(true_sigma) + np.random.normal(scale=0.1, size=2)
            P0 = np.log(true_P + 1e-8) + np.random.normal(scale=0.1, size=(2, 2))
        else:
            beta0 = np.random.uniform(-0.9, 0.9, size=K)
            sigma0 = np.random.normal(size=K)
            P0 = np.random.normal(size=(K, K))
        
        result, params_hat = fit_model(y, beta0, sigma0, P0)
        loglik = -result.fun
        
        if loglik > best_loglik:
            best_loglik = loglik
            best_result = result
            best_params = params_hat
    
    n_params = 2*K + K*(K-1)
    n = len(y)
    aic = -2*best_loglik + 2*n_params
    bic = -2*best_loglik + np.log(n)*n_params
    
    return {
        "K": K,
        "loglik": best_loglik,
        "AIC": aic,
        "BIC": bic,
        "result": best_result,
        "params": best_params
    }

## Simulate data

In [3]:
T = 1000
seed = 1
# AR(1) parameters
beta = np.array([0.2, 0.8])
sigma = np.array([0.5, 1.0])

# transition matrix
P = np.array([
    [0.95, 0.05],
    [0.05, 0.95]
])


y, states = simulate_rs_ar1(T=T, beta=beta, sigma=sigma, P=P, seed=seed)

print("")

## HMM modell med ulikt antall states

Estimerer HMM med 1 til 4 states. Lagrer AIC og BIC, samt likelihood. Bruker guess nær sanne parameter der det er mulig for raskere kode. Printer fremgang

In [4]:
K_vals = [1, 2, 3, 4]
results_all = []

for k in K_vals:
    print(f"\nStarter K = {k}")
    
    best_loglik = -np.inf
    best_result = None
    best_params = None

    if k == 2:
        beta0 = np.log((1 + beta) / (1 - beta)) + np.random.normal(scale=0.1, size=2)
        sigma0 = np.log(sigma) + np.random.normal(scale=0.1, size=2)
        P0 = np.log(P + 1e-8) + np.random.normal(scale=0.1, size=(2, 2))
    else:
        beta0 = np.random.uniform(-0.9, 0.9, size=k)
        sigma0 = np.random.normal(size=k)
        P0 = np.random.normal(size=(k, k))

    result, params_hat = fit_model(y, beta0, sigma0, P0)
    loglik = -result.fun

    print(f"K={k}, loglik={loglik:.2f}")

    if loglik > best_loglik:
        best_loglik = loglik
        best_result = result
        best_params = params_hat

    n_params = 2*k + k*(k-1)
    n = len(y)

    aic = -2*best_loglik + 2*n_params
    bic = -2*best_loglik + np.log(n)*n_params

    print(f"Best for K={k}: AIC={aic:.2f}, BIC={bic:.2f}")

    results_all.append({
        "K": k,
        "loglik": best_loglik,
        "AIC": aic,
        "BIC": bic,
        "result": best_result,
        "params": best_params
    })


Starter K = 1
K=1, loglik=nan
Best for K=1: AIC=inf, BIC=inf

Starter K = 2
K=2, loglik=-1103.58
Best for K=2: AIC=2219.15, BIC=2248.60

Starter K = 3
K=3, loglik=-1099.29
Best for K=3: AIC=2222.57, BIC=2281.47

Starter K = 4
K=4, loglik=-1097.69
Best for K=4: AIC=2235.38, BIC=2333.53


# Undersøke hvordan sample size påvirker

In [5]:
sample_sizes = [200, 500, 1000, 2000]
n_rep = 10
K_vals = [1, 2, 3]

order_results = []

for T in sample_sizes:
    print(f"\n===== Sample size T = {T} =====")
    
    aic_choices = []
    bic_choices = []

    for rep in range(n_rep):
        print(f"Repetisjon {rep+1}/{n_rep}")
        
        # Simuler data fra sann 2-state modell
        y, states = simulate_rs_ar1(
            T=T,
            beta=beta,
            sigma=sigma,
            P=P,
            seed=1000 + rep
        )

        results_all = []

        for k in K_vals:
            print(f"  Estimerer K = {k}")
            
            result_dict = fit_model_robust(y, k, beta, sigma, P, n_starts=3)
            results_all.append(result_dict)

            print(f"    loglik={result_dict['loglik']:.2f}, AIC={result_dict['AIC']:.2f}, BIC={result_dict['BIC']:.2f}")

        best_k_aic = min(results_all, key=lambda d: d["AIC"])["K"]
        best_k_bic = min(results_all, key=lambda d: d["BIC"])["K"]

        aic_choices.append(best_k_aic)
        bic_choices.append(best_k_bic)

        print(f"  Valgt av AIC: K={best_k_aic}")
        print(f"  Valgt av BIC: K={best_k_bic}")

    order_results.append({
        "T": T,
        "AIC_correct_rate": np.mean(np.array(aic_choices) == 2),
        "BIC_correct_rate": np.mean(np.array(bic_choices) == 2),
        "AIC_choices": aic_choices,
        "BIC_choices": bic_choices
    })

print("\n===== Oppsummering =====")
for res in order_results:
    print(
        f"T={res['T']}: "
        f"AIC velger K=2 i {res['AIC_correct_rate']:.2f} av simuleringene, "
        f"BIC velger K=2 i {res['BIC_correct_rate']:.2f} av simuleringene"
    )

# Create DataFrame for plotting
order_df = pd.DataFrame(order_results)


===== Sample size T = 200 =====
Repetisjon 1/10
  Estimerer K = 1
    loglik=-230.15, AIC=464.29, BIC=470.89
  Estimerer K = 2
    loglik=-207.40, AIC=426.81, BIC=446.60
  Estimerer K = 3
    loglik=-205.57, AIC=435.14, BIC=474.72
  Valgt av AIC: K=2
  Valgt av BIC: K=2
Repetisjon 2/10
  Estimerer K = 1
    loglik=-273.06, AIC=550.13, BIC=556.72
  Estimerer K = 2
    loglik=-259.67, AIC=531.34, BIC=551.13
  Estimerer K = 3
    loglik=-253.81, AIC=531.63, BIC=571.21
  Valgt av AIC: K=2
  Valgt av BIC: K=2
Repetisjon 3/10
  Estimerer K = 1
    loglik=-265.09, AIC=534.19, BIC=540.78
  Estimerer K = 2
    loglik=-245.65, AIC=503.30, BIC=523.09
  Estimerer K = 3
    loglik=-239.66, AIC=503.33, BIC=542.91
  Valgt av AIC: K=2
  Valgt av BIC: K=2
Repetisjon 4/10
  Estimerer K = 1
    loglik=-233.76, AIC=471.52, BIC=478.12
  Estimerer K = 2
    loglik=-217.17, AIC=446.34, BIC=466.13
  Estimerer K = 3
    loglik=-214.01, AIC=452.01, BIC=491.59
  Valgt av AIC: K=2
  Valgt av BIC: K=2
Repetisjon 

/Users/adnerolstad/Github/STK-MAT2011/code/methods/hmm_ar_1_k_states.py:94: RuntimeWarning: overflow encountered in exp
  row = np.exp(P_raw[i])
/Users/adnerolstad/Github/STK-MAT2011/code/methods/hmm_ar_1_k_states.py:95: RuntimeWarning: invalid value encountered in divide
  P[i] = row / np.sum(row)


    loglik=-1154.81, AIC=2333.63, BIC=2392.52
  Valgt av AIC: K=2
  Valgt av BIC: K=2
Repetisjon 3/10
  Estimerer K = 1
    loglik=-1229.69, AIC=2463.37, BIC=2473.19
  Estimerer K = 2
    loglik=-1135.55, AIC=2283.10, BIC=2312.55
  Estimerer K = 3
    loglik=-1132.25, AIC=2288.50, BIC=2347.39
  Valgt av AIC: K=2
  Valgt av BIC: K=2
Repetisjon 4/10
  Estimerer K = 1
    loglik=-inf, AIC=inf, BIC=inf
  Estimerer K = 2
    loglik=-1151.29, AIC=2314.59, BIC=2344.03
  Estimerer K = 3
    loglik=-1145.39, AIC=2314.77, BIC=2373.66
  Valgt av AIC: K=2
  Valgt av BIC: K=2
Repetisjon 5/10
  Estimerer K = 1
    loglik=-1243.17, AIC=2490.33, BIC=2500.15
  Estimerer K = 2
    loglik=-1172.12, AIC=2356.25, BIC=2385.69
  Estimerer K = 3
    loglik=-1168.22, AIC=2360.44, BIC=2419.34
  Valgt av AIC: K=2
  Valgt av BIC: K=2
Repetisjon 6/10
  Estimerer K = 1
    loglik=-1254.66, AIC=2513.32, BIC=2523.13
  Estimerer K = 2
    loglik=-1185.52, AIC=2383.05, BIC=2412.49
  Estimerer K = 3
    loglik=-1180.20,

/Users/adnerolstad/Github/STK-MAT2011/code/methods/hmm_ar_1_k_states.py:89: RuntimeWarning: overflow encountered in exp
  sigma = np.exp(sigma_raw)
/Users/adnerolstad/Github/STK-MAT2011/code/methods/hmm_ar_1_k_states.py:86: RuntimeWarning: overflow encountered in exp
  beta = (1-np.exp(-beta_raw)) / (1 + np.exp(-beta_raw))
/Users/adnerolstad/Github/STK-MAT2011/code/methods/hmm_ar_1_k_states.py:86: RuntimeWarning: invalid value encountered in divide
  beta = (1-np.exp(-beta_raw)) / (1 + np.exp(-beta_raw))


    loglik=-2200.21, AIC=4424.42, BIC=4491.63
  Valgt av AIC: K=3
  Valgt av BIC: K=2
Repetisjon 5/10
  Estimerer K = 1
    loglik=-2513.54, AIC=5031.08, BIC=5042.28
  Estimerer K = 2
    loglik=-2389.12, AIC=4790.24, BIC=4823.84
  Estimerer K = 3
    loglik=-2383.73, AIC=4791.45, BIC=4858.67
  Valgt av AIC: K=2
  Valgt av BIC: K=2
Repetisjon 6/10
  Estimerer K = 1
    loglik=-2489.38, AIC=4982.75, BIC=4993.96
  Estimerer K = 2
    loglik=-2328.12, AIC=4668.24, BIC=4701.84
  Estimerer K = 3
    loglik=-2323.06, AIC=4670.13, BIC=4737.34
  Valgt av AIC: K=2
  Valgt av BIC: K=2
Repetisjon 7/10
  Estimerer K = 1
    loglik=-2432.76, AIC=4869.52, BIC=4880.72
  Estimerer K = 2
    loglik=-2237.20, AIC=4486.41, BIC=4520.01
  Estimerer K = 3
    loglik=-2231.80, AIC=4487.61, BIC=4554.82
  Valgt av AIC: K=2
  Valgt av BIC: K=2
Repetisjon 8/10
  Estimerer K = 1
    loglik=-2436.82, AIC=4877.64, BIC=4888.84
  Estimerer K = 2
    loglik=-2276.76, AIC=4565.52, BIC=4599.13
  Estimerer K = 3
    logl

# Litt prompting

In [6]:
beta1 = 0.2
beta2_vals = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

sigma = np.array([0.5, 1.0])

P = np.array([
    [0.95, 0.05],
    [0.05, 0.95]
])

T = 1000
K_vals = [1, 2, 3]

diff_results = []

for beta2 in beta2_vals:
    beta = np.array([beta1, beta2])
    diff = abs(beta2 - beta1)

    print(f"\n===== beta difference = {diff:.2f} =====")
    print(f"beta = {beta}")

    y, states = simulate_rs_ar1(
        T=T,
        beta=beta,
        sigma=sigma,
        P=P,
        seed=1
    )

    results_all = []

    for k in K_vals:
        print(f"  Estimerer K={k}")

        result_dict = fit_model_robust(y, k, beta, sigma, P, n_starts=3)
        results_all.append(result_dict)

        print(f"    loglik={result_dict['loglik']:.2f}, AIC={result_dict['AIC']:.2f}, BIC={result_dict['BIC']:.2f}")

    best_k_aic = min(results_all, key=lambda d: d["AIC"])["K"]
    best_k_bic = min(results_all, key=lambda d: d["BIC"])["K"]

    print(f"  → AIC velger K={best_k_aic}")
    print(f"  → BIC velger K={best_k_bic}")

    diff_results.append({
        "beta_diff": diff,
        "best_k_aic": best_k_aic,
        "best_k_bic": best_k_bic
    })

# Create DataFrame for plotting
diff_df = pd.DataFrame(diff_results)


===== beta difference = 0.10 =====
beta = [0.2 0.3]
  Estimerer K=1
    loglik=-1141.88, AIC=2287.76, BIC=2297.58
  Estimerer K=2
    loglik=-1098.88, AIC=2209.76, BIC=2239.21
  Estimerer K=3
    loglik=-1096.33, AIC=2216.65, BIC=2275.55
  → AIC velger K=2
  → BIC velger K=2

===== beta difference = 0.20 =====
beta = [0.2 0.4]
  Estimerer K=1
    loglik=-1143.46, AIC=2290.93, BIC=2300.74
  Estimerer K=2
    loglik=-1098.15, AIC=2208.29, BIC=2237.74
  Estimerer K=3
    loglik=-1094.95, AIC=2213.90, BIC=2272.80
  → AIC velger K=2
  → BIC velger K=2

===== beta difference = 0.30 =====
beta = [0.2 0.5]
  Estimerer K=1
    loglik=-1147.38, AIC=2298.76, BIC=2308.58
  Estimerer K=2
    loglik=-1098.22, AIC=2208.44, BIC=2237.88
  Estimerer K=3
    loglik=-1094.62, AIC=2213.25, BIC=2272.14
  → AIC velger K=2
  → BIC velger K=2

===== beta difference = 0.40 =====
beta = [0.2 0.6]
  Estimerer K=1
    loglik=-1154.18, AIC=2312.37, BIC=2322.18
  Estimerer K=2
    loglik=-1099.10, AIC=2210.21, BIC=

In [7]:
# Plot for Sample Size Dependence
fig = go.Figure()
fig.add_trace(go.Scatter(x=order_df['T'], y=order_df['AIC_correct_rate'], mode='lines+markers', name='AIC Correct Rate'))
fig.add_trace(go.Scatter(x=order_df['T'], y=order_df['BIC_correct_rate'], mode='lines+markers', name='BIC Correct Rate'))
fig.update_layout(title='Correct Selection Rate vs Sample Size', xaxis_title='Sample Size (T)', yaxis_title='Correct Rate')
fig.show()

In [8]:
# Plot for Parameter Difference Dependence
fig = go.Figure()
fig.add_trace(go.Scatter(x=diff_df['beta_diff'], y=diff_df['best_k_aic'], mode='lines+markers', name='AIC Chosen K'))
fig.add_trace(go.Scatter(x=diff_df['beta_diff'], y=diff_df['best_k_bic'], mode='lines+markers', name='BIC Chosen K'))
fig.update_layout(title='Chosen K vs Beta Difference', xaxis_title='Beta Difference', yaxis_title='Chosen K')
fig.show()